# Experiment: Case-001 Branch Review Clinician Brief Gate

Objective: verify that the doctor-facing brief stays ordered, narrow, and public-safe after the June branch-review gates.

Success criteria:
- the main paper is never the first clinician-facing artifact;
- packet readiness is checked before any branch-scope question;
- the brief asks one branch-scope question only after packet readiness;
- response capture remains label-only;
- blocked patient-action outputs stay blocked.

In [ ]:
from __future__ import annotations

REQUIRED_PACKET_DOMAINS = [
    "diagnosis_genotype_phenotype",
    "transfusion_burden_response",
    "blood_bank_immune_history",
    "iron_organ_chelation_status",
    "hct_gene_cell_access_context",
    "consent_ethics_private_owner_review",
]

CONVERSATION_ORDER = [
    "send_concise_doctor_handoff_before_main_paper",
    "verify_private_packet_six_domains",
    "ask_one_branch_scope_question_only_if_packet_ready",
    "capture_response_as_public_safe_labels_only",
    "keep_main_paper_as_appendix_only",
]

BLOCKED_OUTPUTS = {
    "patient-specific eligibility claim",
    "therapy selection",
    "trial-screening instruction",
    "referral instruction",
    "travel or import plan",
    "purchase or procurement route",
    "dose or treatment recommendation",
    "transfusion or chelation change",
    "lab contact permission",
    "sample routing",
}

## Brief Sequencing Model

This gate is an agenda check, not a medical decision. The branch question must not run before the private packet is complete enough for clinician review.

In [ ]:
def build_brief_state(ready_domains: set[str]) -> dict[str, object]:
    """Return the public-safe brief state for one packet-readiness scenario."""
    missing = [
        domain for domain in REQUIRED_PACKET_DOMAINS if domain not in ready_domains
    ]
    packet_ready = not missing
    branch_question_state = (
        "branch_question_allowed" if packet_ready else "packet_first"
    )
    return {
        "conversation_order": CONVERSATION_ORDER,
        "missing_domains": missing,
        "packet_ready": packet_ready,
        "branch_question_state": branch_question_state,
        "blocked_outputs": sorted(BLOCKED_OUTPUTS),
    }


packet_incomplete = build_brief_state({"diagnosis_genotype_phenotype"})
packet_ready = build_brief_state(set(REQUIRED_PACKET_DOMAINS))

assert packet_incomplete["branch_question_state"] == "packet_first"
assert packet_ready["branch_question_state"] == "branch_question_allowed"
assert (
    packet_incomplete["conversation_order"][0]
    == "send_concise_doctor_handoff_before_main_paper"
)
assert packet_ready["conversation_order"][-1] == "keep_main_paper_as_appendix_only"

{
    "incomplete_state": packet_incomplete["branch_question_state"],
    "ready_state": packet_ready["branch_question_state"],
    "required_domains": len(REQUIRED_PACKET_DOMAINS),
}

## Blocked Output Check

The brief can ask what is missing and who should review next. It must not produce treatment, referral, trial, travel, import, lab-contact, dose, transfusion, chelation, or sample-routing actions.

In [ ]:
expected_blockers = {
    "therapy selection",
    "trial-screening instruction",
    "referral instruction",
    "dose or treatment recommendation",
    "transfusion or chelation change",
    "sample routing",
}

assert expected_blockers.issubset(BLOCKED_OUTPUTS)
assert "ask_one_branch_scope_question_only_if_packet_ready" in CONVERSATION_ORDER
assert CONVERSATION_ORDER.index(
    "verify_private_packet_six_domains"
) < CONVERSATION_ORDER.index("ask_one_branch_scope_question_only_if_packet_ready")

result = {
    "sequencing_passed": True,
    "blocked_output_count": len(BLOCKED_OUTPUTS),
    "decision": "continue_with_concise_clinician_brief",
}
result

## Result

The useful clinician-facing artifact is not the main paper. It is a concise brief that asks packet-readiness first, one branch-scope question second, and response-label capture third. The notebook supports updating the doctor handoff PDF source and public case-context note without changing clinical state.